# analysis_overview_dashboard_schedule_v1

验证 `src/analysis` 生成的 regime/window 模型差异 overview 能被 dashboard 和 schedule 复用。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analysis import (
    analyze_model_effects_by_regime_and_window,
    build_analysis_overview,
    build_regime_window_analysis_report,
    compare_models,
    compute_regime_metrics,
    compute_window_metrics,
    write_analysis_report,
)
from src.core import DatasetSnapshot
from src.dashboard import build_dashboard_view_model, write_dashboard_html, write_model_matrix_dashboard_export
from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

EXPERIMENT = "analysis_overview_dashboard_schedule_v1"
CARD = ROOT / "research" / EXPERIMENT
ANALYSIS_CARD = CARD / "analysis_card"
CARD.mkdir(parents=True, exist_ok=True)
ANALYSIS_CARD.mkdir(parents=True, exist_ok=True)

In [2]:
dates = list(pd.date_range("2026-01-01", periods=8, freq="D")) * 2
regimes = ["bull", "bull", "bull", "neutral", "stress", "stress", "stress", "downtrend"] * 2
daily = pd.DataFrame(
    {
        "date": dates,
        "model": ["ridge_mse__all_features"] * 8 + ["gcn_mse__top_ic_factors"] * 8,
        "regime": regimes,
        "nav": [
            1.00,
            1.02,
            1.04,
            1.03,
            1.01,
            0.99,
            1.00,
            1.01,
            1.00,
            1.04,
            1.08,
            1.07,
            1.02,
            0.96,
            0.97,
            0.98,
        ],
        "benchmark_nav": [1.0, 1.01, 1.02, 1.02, 1.00, 0.99, 1.00, 1.00] * 2,
        "position": [0.2, 0.3, 0.5, 0.1, 0.4, 0.2, 0.1, 0.2] * 2,
        "turnover": [0.0, 0.1, 0.2, 0.4, 0.3, 0.1, 0.1, 0.2] * 2,
    }
)
windows = {
    "early": ("2026-01-01", "2026-01-04"),
    "stress_window": ("2026-01-05", "2026-01-07"),
}
baseline_model = "ridge_mse__all_features"

effect = analyze_model_effects_by_regime_and_window(daily, windows=windows, baseline_model=baseline_model)
overview = build_analysis_overview(effect)
report = build_regime_window_analysis_report(
    artifact_id=EXPERIMENT,
    effect_analysis=effect,
    model_comparison=compare_models(daily),
    regime_metrics=compute_regime_metrics(daily),
    window_metrics=compute_window_metrics(daily, windows=windows),
    summary={
        "status": "analysis_overview_completed",
        "analysis_summary_zh": "验证 analysis_overview 可统一服务 dashboard 总览和 schedule 通知。",
    },
)
dataset = DatasetSnapshot(
    name="synthetic_regime_window_overview",
    universe=("SOXL", "SOXX"),
    start="2026-01-01",
    end="2026-01-08",
    calendar_path="data/qlib/soxl/calendars/day.txt",
    data_paths=("synthetic_daily_frame",),
)
export_result = write_analysis_report(report, ANALYSIS_CARD, dataset=dataset)
dashboard_payload = write_model_matrix_dashboard_export(
    ANALYSIS_CARD,
    output_path=CARD / "dashboard_export.json",
    top_n=5,
)
view_model = build_dashboard_view_model(dashboard_payload)
dashboard_html = write_dashboard_html(view_model, output_path=CARD / "dashboard.html", title="SOXL Analysis Overview")

overview

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'analysis.overview.v1',
 'baseline_model': 'ridge_mse__all_features',
 'model_count': 2,
 'regime_count': 4,
 'window_count': 2,
 'window_regime_count': 3,
 'segment_effect_matrix_count': 10,
 'segment_types': ['all', 'regime', 'window', 'window_regime'],
 'qlib_metric_source': 'qlib.contrib.evaluate.risk_analysis',
 'review_signal_counts': {'strength': 1, 'weakness': 4},
 'decision_hint_counts': {'block_or_rework': 4, 'promote_candidate': 1},
 'top_review': {'model': 'gcn_mse__top_ic_factors',
  'baseline_model': 'ridge_mse__all_features',
  'segment_type': 'regime',
  'segment': 'stress',
  'window': '',
  'regime': 'stress',
  'review_signal': 'weakness',
  'decision_hint': 'block_or_rework',
  'priority_rank': 1,
  'priority_score': 78.04619,
  'relative_return_pct_delta': -3.9118617744127393,
  'stability_score_delta': -68.27138855380323,
  'max_drawdown_pct_delta': 3.90215492137449,
  'review_note': 'gcn_mse__top_ic_factors 在 stress 弱于基准 ridge_mse__all_features

In [3]:
events: list[str] = []


def matrix_runner(**kwargs):
    return {"variant_count": 2, "status": "analysis_overview_completed"}


def dashboard_exporter(card_dir: Path, *, output_path: Path, top_n: int):
    return write_model_matrix_dashboard_export(ANALYSIS_CARD, output_path=output_path, top_n=top_n)


config = ModelDashboardPipelineConfig(
    name="model_dashboard_refresh",
    card_dir=ANALYSIS_CARD,
    dashboard_output_path=CARD / "scheduled_dashboard_export.json",
    start="2026-01-01",
    end="2026-01-08",
    train_end="2026-01-04",
    model_names=("ridge_mse", "gcn_mse"),
    notify_details=True,
)
schedule_result = run_model_dashboard_refresh_once(
    config,
    lock_dir=CARD / "locks",
    notify=events.append,
    matrix_runner=matrix_runner,
    dashboard_exporter=dashboard_exporter,
)

summary = {
    "schema_version": "research.analysis_overview_dashboard_schedule_v1",
    "status": schedule_result.status,
    "analysis_overview": overview,
    "dashboard_analysis_overview": dashboard_payload["analysis_overview"],
    "view_model_analysis_overview": view_model["analysis_overview"],
    "schedule_payload": schedule_result.payload,
    "notification": events[0] if events else "",
    "artifacts": {
        "analysis_card": str(ANALYSIS_CARD),
        "dashboard_export": str(CARD / "dashboard_export.json"),
        "scheduled_dashboard_export": str(CARD / "scheduled_dashboard_export.json"),
        "dashboard_html": str(CARD / "dashboard.html"),
    },
}
(CARD / "analysis_overview_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
(CARD / "notification_events.json").write_text(json.dumps(events, ensure_ascii=False, indent=2), encoding="utf-8")

summary

{'schema_version': 'research.analysis_overview_dashboard_schedule_v1',
 'status': 'success',
 'analysis_overview': {'schema_version': 'analysis.overview.v1',
  'baseline_model': 'ridge_mse__all_features',
  'model_count': 2,
  'regime_count': 4,
  'window_count': 2,
  'window_regime_count': 3,
  'segment_effect_matrix_count': 10,
  'segment_types': ['all', 'regime', 'window', 'window_regime'],
  'qlib_metric_source': 'qlib.contrib.evaluate.risk_analysis',
  'review_signal_counts': {'strength': 1, 'weakness': 4},
  'decision_hint_counts': {'block_or_rework': 4, 'promote_candidate': 1},
  'top_review': {'model': 'gcn_mse__top_ic_factors',
   'baseline_model': 'ridge_mse__all_features',
   'segment_type': 'regime',
   'segment': 'stress',
   'window': '',
   'regime': 'stress',
   'review_signal': 'weakness',
   'decision_hint': 'block_or_rework',
   'priority_rank': 1,
   'priority_score': 78.04619,
   'relative_return_pct_delta': -3.9118617744127393,
   'stability_score_delta': -68.2713

In [4]:
assert overview["schema_version"] == "analysis.overview.v1"
assert overview["top_weakness"]["model"] == "gcn_mse__top_ic_factors"
assert "stress" in overview["top_weakness"]["segment"]
assert overview["largest_relative_return_spread_segments"]
assert dashboard_payload["analysis_overview"]["schema_version"] == "analysis.overview.v1"
assert view_model["analysis_overview"]["schema_version"] == "analysis.overview.v1"
assert "Analysis Overview" in dashboard_html
assert "regime_gap=" in summary["notification"]
assert summary["schedule_payload"]["analysis_overview_schema_version"] == "analysis.overview.v1"
assert (CARD / "analysis_overview_summary.json").exists()
assert (CARD / "dashboard.html").exists()

## 中文结论

- 做了什么：新增并验证 `analysis_overview`，把 `src/analysis` 的 regime/window 模型差异压缩成 dashboard 和 schedule 可复用的稳定摘要。
- 核心指标：overview schema 为 `analysis.overview.v1`；识别到 `gcn_mse__top_ic_factors` 在 stress 相关分段为 top weakness；schedule 通知包含 `regime_gap=` 摘要。
- regime 差异：`gcn_mse__top_ic_factors` 在 early/bull 分段更强，但在 stress/stress_window 分段弱于 `ridge_mse__all_features`，说明模型差异需要按 regime/window 看，而不是只看全区间。
- 是否晋升：晋升的是 analysis/dashboard/schedule 框架能力；不晋升任何交易策略或模型。
- 风险：本 notebook 使用合成数据验证接口契约，不代表真实 SOXL 表现；真实评估仍需接入 qlib recorder 或模型矩阵产物。
- 下一步：把 overview 接入真实 QuantBench/qlib recorder 卡片的看板刷新，并补更多窗口如 YTD、1/3/6/12 个月。